# GPAT

Gridded Plume Analysis Tool (GPAT) modelling framework. This simulates flight trajectories, estimates fuel burn and emissions, models dispersion effects, and aggregates plume data to a common Eulerian grid for further photochemical and microphysical processing.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from dataclasses import asdict
from pycontrails.models.gpat.gpat import GPAT, SimParams, FlParams, PlumeParams, MetParams, ChemParams, dict_to_dataclass
import os
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [2]:
# global simulation parameters
sim_params = {
    "t_fl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=1)),# (start time, time step, run time)
    "t_pl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=2)),# (start time, time step, max age)
    "t_sim": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(seconds=20), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "lat_bounds": (0.0, 1.0),  # lat bounds [deg]
    "lon_bounds": (0.0, 1.0),  # lon bounds [deg]
    "alt_bounds": (10000, 11000),  # alt bounds [m]
    "hres_sim_c": 0.05,  # coarse horizontal resolution [deg]
    "vres_sim_c": 500,  # coarse vertical resolution [m]
    "hres_sim_f": 0.001,  # fine horizontal resolution [deg]
    "vres_sim_f": 100,  # fine vertical resolution [m]

    "run_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/",
    "data_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/", # "/projects/Impact_of_aviation_on_climate
    "job_id": "GPAT_Dec_2025_test_1_ac",
}

In [3]:
# flight trajectory parameters
fl_params = {
    "mode": "synthetic",
    "file": None,  # flight trajectory file

    "ac_type": "A320",  # aircraft type
    "fl0_speed": 100.0,  # m/s
    "fl0_heading": 45.0,  # deg
    "fl0_coords0": (0.1, 0.1, 10500),  # lat, lon, alt [deg, deg, m]
    "sep_dist": (10000, 5000, 0),  # dx, dy, dz [m]
    "n_ac": 1,  # number of aircraft
}

In [4]:
# plume dispersion parameters
plume_params = {
    "depth": 50.0,  # initial plume depth, [m]
    "width": 50.0,  # initial plume width, [m]
    "verbose_outputs": False,  # print verbose outputs
    "n_slices": 5,  # number of slices in the plume
    "shear": 0.01,  # shear [m/s]
    }

In [5]:
# meteorology parameters
met_params = {
    "eastward_wind": 5.0,  # m/s
    "northward_wind": 3.0,  # m/s
    "lagrangian_tendency_of_air_pressure": 0.0,  # m/s
}

In [6]:
# chemistry parameters
chem_params = {
    "run_chem": True,
    "species_emi": ("NO",),
    "species_plume": ("NO", "NO2", "O3", "NO3", "N2O5",
                      "HNO3", "HONO", "HO2NO2","PAN", 
                      "CH3O2NO2","H2O2", "CH3OOH",
                      "CO", "CH4", "HCHO", "SO2", "SA"),
    "species_out": ("O3", "NO2", "NO", "NO3", "N2O5", 
                    "HNO3", "HONO", "HO2", "OH", "H2O2",
                    "CO", "CH4", "CH3O2","HO2NO2", "PAN", "SO2" )
}

In [7]:
sim_params = SimParams(**sim_params)
fl_params = FlParams(**fl_params)
plume_params = PlumeParams(**plume_params)
met_params = MetParams(**met_params)
chem_params = ChemParams(**chem_params)

gpat = GPAT(sim_params, fl_params, plume_params, met_params, chem_params)


In [8]:
gpat.preprocess_gpat()

flight 0 done


/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:589: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i].dataframe[column] = fl[i].dataframe[column].fillna(method="ffill")
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:642: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fl[i][column] = fl[i][column].fillna(method="ffill")


Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Dec_2025_test_1_ac/boxm_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Dec_2025_test_1_ac/fl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Dec_2025_test_1_ac/pl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Dec_2025_test_1_ac/boxm_out.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Dec_2025_test_1_ac/patch_table.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Dec_2025_test_1_ac/pl_out.nc


/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:1030: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  pl_ds = xr.open_dataset(f"{self.gpat.inputs_job}/pl_ds.nc")


In [9]:
fl_ds = xr.open_dataset(f"{gpat.inputs_job}/fl_ds.nc")

fl_ds

<xarray.Dataset> Size: 5kB
Dimensions:               (flight_id: 1, waypoint: 24)
Coordinates:
  * flight_id             (flight_id) float32 4B 0.0
  * waypoint              (waypoint) int64 192B 0 1 2 3 4 5 ... 19 20 21 22 23
Data variables: (12/25)
    longitude             (flight_id, waypoint) float64 192B ...
    latitude              (flight_id, waypoint) float64 192B ...
    altitude              (flight_id, waypoint) float64 192B ...
    time                  (flight_id, waypoint) datetime64[ns] 192B ...
    true_airspeed         (flight_id, waypoint) float64 192B ...
    aircraft_mass         (flight_id, waypoint) float64 192B ...
    ...                    ...
    HCHO                  (flight_id, waypoint) float64 192B ...
    CH3CHO                (flight_id, waypoint) float64 192B ...
    C2H4                  (flight_id, waypoint) float64 192B ...
    C3H6                  (flight_id, waypoint) float64 192B ...
    C2H2                  (flight_id, waypoint) float64 192B ...
    BENZENE               (flight_id, waypoint) float64 192B ...
Attributes:
    description:  Flight trajectory and emissions data for BOXM

In [10]:
pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")
pl_ds
# pl_ds["emi_species_mass"].sel(species="NO").isel(time=10).values

/tmp/ipykernel_131003/4172039868.py:1: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")


<xarray.Dataset> Size: 420kB
Dimensions:           (flight_id: 1, waypoint: 23, time: 134, species: 1)
Coordinates:
  * flight_id         (flight_id) float32 4B 0.0
  * waypoint          (waypoint) int64 184B 0 1 2 3 4 5 6 ... 17 18 19 20 21 22
  * time              (time) datetime64[ns] 1kB 2022-01-20T13:00:00 ... 2022-...
  * species           (species) <U2 8B 'NO'
Data variables: (12/17)
    fuel_flow         (flight_id, waypoint, time) float64 25kB ...
    fuel_burn         (flight_id, waypoint, time) float64 25kB ...
    true_airspeed     (flight_id, waypoint, time) float64 25kB ...
    age               (flight_id, waypoint, time) timedelta64[ns] 25kB ...
    longitude         (flight_id, waypoint, time) float64 25kB ...
    latitude          (flight_id, waypoint, time) float64 25kB ...
    ...                ...
    sigma_yz          (flight_id, waypoint, time) float64 25kB ...
    sigma_zz          (flight_id, waypoint, time) float64 25kB ...
    sin_a             (flight_id, waypoint, time) float64 25kB ...
    cos_a             (flight_id, waypoint, time) float64 25kB ...
    altitude          (flight_id, waypoint, time) float64 25kB ...
    emi_species_mass  (flight_id, waypoint, time, species) float64 25kB ...
Attributes:
    ts_fl:          60.0
    ts_pl:          60.0
    ts_sim:         20.0
    species_emi:    NO
    species_plume:  ['NO', 'NO2', 'O3', 'NO3', 'N2O5', 'HNO3', 'HONO', 'HO2NO...

In [11]:
pl_ds["emi_species_mass"].sel(species="NO").isel(flight_id=0, waypoint=0, time=slice(0,10)).values

array([1.08379147, 1.08379147, 1.08379147, 1.08379147, 1.08379147,
       1.08379147, 1.08379147, 1.08379147, 1.08379147, 1.08379147])

In [12]:
gpat.pl

,flight_id,waypoint,fuel_flow,fuel_burn,true_airspeed,CO2,H2O,SO2,NO,NO2,...,level,width,depth,heading,sigma_yy,sigma_yz,sigma_zz,sin_a,cos_a,altitude
0,0.0,0,1.211896,72.713735,100.230055,229.775402,89.437894,0.061080,1.083791,0.057042,...,244.739952,117.644397,109.552274,NaN,1.730026e+03,8.250638e+02,1500.212595,NaN,NaN,10500.0
1,0.0,0,1.211896,72.713735,100.230055,229.775402,89.437894,0.061080,1.083791,0.057042,...,244.739952,162.239393,118.335541,45.866007,3.290203e+03,1.800251e+03,1750.412526,0.717713,0.696339,10500.0
120,0.0,1,1.211905,72.714320,100.229974,229.777250,89.438613,0.061080,1.083808,0.057043,...,244.739952,117.643185,109.541429,45.866007,1.729990e+03,8.249747e+02,1499.915578,0.717713,0.696339,10500.0
2,0.0,0,1.211896,72.713735,100.230055,229.775402,89.437894,0.061080,1.083791,0.057042,...,244.739952,221.100432,126.510072,45.866006,6.110675e+03,2.925555e+03,2000.599787,0.717713,0.696339,10500.0
121,0.0,1,1.211905,72.714320,100.229974,229.777250,89.438613,0.061080,1.083808,0.057043,...,244.739952,162.232362,118.315457,45.866006,3.289917e+03,1.799895e+03,1749.818418,0.717713,0.696339,10500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,0.0,13,1.212071,72.724242,100.227267,229.808604,89.450817,0.061088,1.084082,0.057057,...,244.739952,21183.912456,493.645597,45.866924,5.609477e+07,1.133233e+06,30460.746974,0.717724,0.696327,10500.0
1797,0.0,14,1.212089,72.725311,100.226896,229.811982,89.452132,0.061089,1.084111,0.057058,...,244.739952,20915.932725,491.358964,45.867115,5.468453e+07,1.113728e+06,30179.203932,0.717727,0.696325,10500.0
1679,0.0,13,1.212071,72.724242,100.227267,229.808604,89.450817,0.061088,1.084082,0.057057,...,244.739952,21441.201964,495.623936,45.867108,5.746564e+07,1.151582e+06,30705.385793,0.717727,0.696325,10500.0
1798,0.0,14,1.212089,72.725311,100.226896,229.811982,89.452132,0.061089,1.084111,0.057058,...,244.739952,21172.037827,493.344023,45.867108,5.603190e+07,1.131909e+06,30423.540635,0.717727,0.696325,10500.0


In [13]:
pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")
print(pl_ds["emi_species_mass"].dims)   # Should print: ('flight_id', 'waypoint', 'time', 'species')
print(pl_ds["emi_species_mass"].shape)  # e.g., (1, 24, 121, 12) for 1 flight, 24 waypoints, 121 timesteps, 12 species

('flight_id', 'waypoint', 'time', 'species')
(1, 23, 134, 1)


/tmp/ipykernel_131003/2877652587.py:1: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")


In [20]:
boxm_ds = xr.open_dataset(f"{gpat.inputs_job}/boxm_ds.nc")

In [21]:
gpat.boxm_ds_stacked

<xarray.Dataset> Size: 29MB
Dimensions:          (time: 721, cell: 800, species: 219)
Coordinates:
  * time             (time) datetime64[ns] 6kB 2022-01-20T12:00:00 ... 2022-0...
    air_pressure     (cell) float64 6kB 2.354e+04 2.354e+04 ... 2.544e+04
    altitude         (cell) float64 6kB 1.075e+04 1.075e+04 ... 1.025e+04
  * species          (species) <U10 9kB 'O1D' 'O' 'OH' ... 'EMPOA' 'P2007'
    level            (cell) float64 6kB 235.4 235.4 235.4 ... 254.4 254.4 254.4
    longitude        (cell) float64 6kB 0.025 0.025 0.025 ... 0.975 0.975 0.975
    latitude         (cell) float64 6kB 0.025 0.075 0.125 ... 0.875 0.925 0.975
Dimensions without coordinates: cell
Data variables:
    air_temperature  (time, cell) float64 5MB 226.8 226.8 226.8 ... 231.5 231.5
    H2O              (time, cell) float64 5MB 2.936e+15 2.93e+15 ... 3.788e+15
    M                (time, cell) float64 5MB 7.518e+18 7.518e+18 ... 7.959e+18
    O2               (time, cell) float64 5MB 1.563e+18 1.563e+18 ... 1.655e+18
    N2               (time, cell) float64 5MB 5.87e+18 5.87e+18 ... 6.215e+18
    sza              (time, cell) float64 5MB 0.3539 0.3548 ... 1.061 1.062
    bg_chem          (species, cell) float64 1MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes: (12/15)
    ts_fl:              60.0
    ts_pl:              60.0
    ts_sim:             20.0
    hres_sim_c:         0.05
    vres_sim_c:         500
    hres_sim_f:         0.001
    ...                 ...
    species_out:        ('O3', 'NO2', 'NO', 'NO3', 'N2O5', 'HNO3', 'HONO', 'H...
    species_emi_num:    [8]
    species_plume_num:  [  8   4   6   5   7  14  13  15 198 217  12 144  11 ...
    species_out_num:    [  6   4   8   5   7  14  13   9   3  12  11  21  22 ...
    description:        BOXM coarse-grid meteorology and background chemistry...
    note:               Emissions and plume segments handled separately via P...

In [17]:
gpat.boxm_out_stacked

<xarray.Dataset> Size: 148MB
Dimensions:      (time: 721, species_out: 16, cell: 800)
Coordinates:
  * time         (time) datetime64[ns] 6kB 2022-01-20T12:00:00 ... 2022-01-20...
  * species_out  (species_out) <U10 640B 'O3' 'NO2' 'NO' ... 'PAN' 'SO2'
    level        (cell) float64 6kB 254.4 254.4 254.4 ... 235.4 235.4 235.4
    longitude    (cell) float64 6kB 0.025 0.025 0.025 ... 0.975 0.975 0.975
    latitude     (cell) float64 6kB 0.025 0.075 0.125 ... 0.875 0.925 0.975
Dimensions without coordinates: cell
Data variables:
    Y_bg_c       (time, species_out, cell) float64 74MB dask.array<chunksize=(721, 16, 800), meta=np.ndarray>
    Y_del_c      (time, species_out, cell) float64 74MB dask.array<chunksize=(721, 16, 800), meta=np.ndarray>
    active_flag  (time, cell) bool 577kB dask.array<chunksize=(721, 800), meta=np.ndarray>

In [18]:
gpat.patch_table

<xarray.Dataset> Size: 680B
Dimensions:        (row: 0, species_plume: 17)
Coordinates:
  * row            (row) int64 0B 
    time           (row) datetime64[ns] 0B 
    patch_id       (row) int64 0B 
    latitude       (row) float64 0B 
    longitude      (row) float64 0B 
    level          (row) int64 0B 
    latitude_f     (row) float64 0B 
    longitude_f    (row) float64 0B 
    level_f        (row) int64 0B 
  * species_plume  (species_plume) <U10 680B 'NO' 'NO2' 'O3' ... 'SO2' 'SA'
Data variables:
    Y_del_f        (row, species_plume) float64 0B dask.array<chunksize=(0, 17), meta=np.ndarray>

In [19]:
gpat.pl_out.sel(waypoint=10)

<xarray.Dataset> Size: 38kB
Dimensions:             (flight_id: 1, time: 134, species_plume: 17)
Coordinates:
  * flight_id           (flight_id) float32 4B 0.0
    waypoint            int64 8B 10
  * time                (time) datetime64[ns] 1kB 2022-01-20T13:00:00 ... 202...
  * species_plume       (species_plume) <U10 680B 'NO' 'NO2' 'O3' ... 'SO2' 'SA'
Data variables: (12/18)
    fuel_flow           (flight_id, time) float64 1kB ...
    fuel_burn           (flight_id, time) float64 1kB ...
    true_airspeed       (flight_id, time) float64 1kB ...
    age                 (flight_id, time) timedelta64[ns] 1kB ...
    longitude           (flight_id, time) float64 1kB ...
    latitude            (flight_id, time) float64 1kB ...
    ...                  ...
    sigma_zz            (flight_id, time) float64 1kB ...
    sin_a               (flight_id, time) float64 1kB ...
    cos_a               (flight_id, time) float64 1kB ...
    altitude            (flight_id, time) float64 1kB ...
    delta_species_mass  (flight_id, time, species_plume) float64 18kB 0.0 ......
    aspect_ratio        (flight_id, time) float64 1kB nan nan nan ... nan nan
Attributes:
    ts_fl:          60.0
    ts_pl:          60.0
    ts_sim:         20.0
    species_emi:    NO
    species_plume:  ['NO', 'NO2', 'O3', 'NO3', 'N2O5', 'HNO3', 'HONO', 'HO2NO...